In [1]:
import tensorflow as tf
from tensorflow.keras.preprocessing.image import ImageDataGenerator

In [ ]:
# Training data augmentation
train_datagen = ImageDataGenerator(
    rescale=1./255,
    rotation_range=10,
    zoom_range=0.1,
    horizontal_flip=True
)

training_set = train_datagen.flow_from_directory(
    './data/train',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)

# Validation data (no augmentation, only rescaling)
valid_datagen = ImageDataGenerator(rescale=1./255)

validation_set = valid_datagen.flow_from_directory(
    './data/val',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary'
)


# Test data (no augmentation, no shuffle!)
test_datagen = ImageDataGenerator(rescale=1./255)

test_set = test_datagen.flow_from_directory(
    './data/test',
    target_size=(224, 224),
    batch_size=32,
    class_mode='binary',
    shuffle=False
)

Found 6800 images belonging to 2 classes.
Found 1700 images belonging to 2 classes.
Found 30 images belonging to 2 classes.


In [4]:
cnn = tf.keras.models.Sequential()

# First Convolution + Pooling
cnn.add(tf.keras.layers.Conv2D(filters=32, kernel_size=3, activation='relu', input_shape=(224, 224, 3)))
cnn.add(tf.keras.layers.MaxPool2D(pool_size=2, strides=2))

# Flatten
cnn.add(tf.keras.layers.Flatten())

# Fully connected layers
cnn.add(tf.keras.layers.Dense(units=128, activation='relu'))

# Dropout to reduce overfitting
cnn.add(tf.keras.layers.Dropout(0.5))

# Output layer
cnn.add(tf.keras.layers.Dense(units=1, activation='sigmoid'))

/opt/homebrew/anaconda3/lib/python3.11/site-packages/keras/src/layers/convolutional/base_conv.py:107: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [5]:
cnn.compile(optimizer = 'adam', loss = 'binary_crossentropy', metrics = ['accuracy'])

In [ ]:
history = cnn.fit(training_set, validation_data=validation_set, epochs=5)

/opt/homebrew/anaconda3/lib/python3.11/site-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


Epoch 1/5
 70/213 ━━━━━━━━━━━━━━━━━━━━ 1:38:32 41s/step - accuracy: 0.5862 - loss: 10.4087

In [ ]:
test_loss, test_acc = cnn.evaluate(test_set)
print("Test accuracy:", test_acc)

4/4 ━━━━━━━━━━━━━━━━━━━━ 2s 331ms/step - accuracy: 0.8727 - loss: 0.3118
Test accuracy: 0.8727272748947144


In [ ]:
cnn.save('xray_cnn_model.keras')